# Biohub - Cell Tracking During Development
## Score: 0.810

## Configuration

In [ ]:
import os

COMP_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
TEST_DIR = f"{COMP_DIR}/test"
ARTIFACTS = (
    "/kaggle/input/datasets/thibautgoldsborough/"
    "cellmot-baseline-artifacts/cellmot-baseline-artifacts"
)
REPO_DIR = "/kaggle/working/repo"
METHOD = "unet_transformer"
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
DET_THRESHOLD = 0.99
UNET_BATCH_SIZE = 4
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0
GAP_MAX_UM = 6.0
DIVISION_PARENT_MAX_UM = 7.0
SISTER_MAX_UM = 12.0
MIN_TRACK_LEN = 3
OUTPUT_PATH = "/kaggle/working/submission.csv"

COMP_DIR, ARTIFACTS, OUTPUT_PATH

## Offline Install

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-index",
        "--find-links",
        f"{ARTIFACTS}/wheels",
        "--upgrade-strategy",
        "only-if-needed",
        "tracksdata",
        "zarr>=3.0.10",
        "pyscipopt",
    ],
    check=True,
)

shutil.copytree(f"{ARTIFACTS}/repo", REPO_DIR, dirs_exist_ok=True)
shutil.copytree(f"{ARTIFACTS}/weights", f"{REPO_DIR}/weights", dirs_exist_ok=True)
sys.path.insert(0, f"{REPO_DIR}/src")

sorted(Path(REPO_DIR, "weights", METHOD, "split_0").iterdir())

## Test Split

In [ ]:
import json
from pathlib import Path

test_stems = sorted(
    path.name.replace(".zarr", "")
    for path in Path(TEST_DIR).glob("*.zarr")
)
splits_path = Path(REPO_DIR) / "kaggle_test_splits.json"
splits_path.write_text(
    json.dumps([{"split": 0, "train": [], "test": test_stems}])
)
len(test_stems), test_stems[:5]

## Inference

In [ ]:
import os
import subprocess

cmd = [
    sys.executable,
    "scripts/predict_unet_transformer.py",
    "--data-dir",
    TEST_DIR,
    "--splits",
    "kaggle_test_splits.json",
    "--split",
    "0",
    "--weights",
    WEIGHTS,
    "--unet-batch-size",
    str(UNET_BATCH_SIZE),
    "--det-threshold",
    str(DET_THRESHOLD),
    "--ilp-edge-weight",
    str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight",
    str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight",
    str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight",
    str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    cmd.append("--use-ilp")

print(" ".join(cmd))
subprocess.run(
    cmd,
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": "src"},
    check=True,
)

## Graph Repair

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

pred_dirs = sorted(Path(REPO_DIR, "predictions").glob(f"*/{METHOD}/split_0"))
assert pred_dirs, "No prediction directory found"
PRED_DIR = pred_dirs[0]
REPAIR_DIR = Path("/kaggle/working/repaired_geffs")
REPAIR_DIR.mkdir(parents=True, exist_ok=True)

repair_code = r"""
import csv
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import zarr
from scipy.optimize import linear_sum_assignment

pred_dir = Path(sys.argv[1])
out_dir = Path(sys.argv[2])
gap_max_um = float(sys.argv[3])
div_parent_max_um = float(sys.argv[4])
sister_max_um = float(sys.argv[5])
min_track_len = int(sys.argv[6])
scale = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)
out_dir.mkdir(parents=True, exist_ok=True)


def load_geff(path):
    root = zarr.open(str(path), mode="r")
    node_ids = np.asarray(root["nodes"]["ids"][:])
    t = np.asarray(root["nodes"]["props"]["t"]["values"][:])
    z = np.asarray(root["nodes"]["props"]["z"]["values"][:])
    y = np.asarray(root["nodes"]["props"]["y"]["values"][:])
    x = np.asarray(root["nodes"]["props"]["x"]["values"][:])
    edges = np.asarray(root["edges"]["ids"][:])
    node_keep = None
    edge_keep = None
    if "solution" in root["nodes"]["props"]:
        node_keep = np.asarray(root["nodes"]["props"]["solution"]["values"][:]).astype(bool)
    if "solution" in root["edges"]["props"]:
        edge_keep = np.asarray(root["edges"]["props"]["solution"]["values"][:]).astype(bool)
    nodes = {}
    for i, node_id in enumerate(node_ids):
        if node_keep is not None and not node_keep[i]:
            continue
        nodes[int(node_id)] = (
            int(t[i]),
            float(z[i]),
            float(y[i]),
            float(x[i]),
        )
    links = []
    for i, (source, target) in enumerate(edges):
        if edge_keep is not None and not edge_keep[i]:
            continue
        source, target = int(source), int(target)
        if source in nodes and target in nodes:
            links.append((source, target))
    return nodes, links


def save_csv_graph(path, nodes, links):
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["node_id", "t", "z", "y", "x"])
        for node_id, (t, z, y, x) in sorted(nodes.items()):
            writer.writerow([node_id, t, z, y, x])
        writer.writerow(["source_id", "target_id"])
        for source, target in links:
            writer.writerow([source, target])


def dist_um(a, b):
    delta = (np.asarray(a[1:], dtype=np.float64) - np.asarray(b[1:], dtype=np.float64)) * scale
    return float(np.sqrt((delta ** 2).sum()))


def hungarian_pairs(left_ids, right_ids, nodes, max_um):
    if not left_ids or not right_ids:
        return []
    left = np.asarray([nodes[i][1:] for i in left_ids], dtype=np.float64) * scale
    right = np.asarray([nodes[i][1:] for i in right_ids], dtype=np.float64) * scale
    d2 = ((left[:, None, :] - right[None, :, :]) ** 2).sum(axis=2)
    cost = d2.copy()
    max_d2 = max_um ** 2
    cost[d2 > max_d2] = 1e6
    rr, cc = linear_sum_assignment(cost)
    pairs = []
    for i, j in zip(rr, cc):
        if cost[i, j] < 1e6:
            pairs.append((left_ids[i], right_ids[j]))
    return pairs


def gap_close(nodes, links):
    outgoing = defaultdict(list)
    incoming = defaultdict(list)
    for source, target in links:
        outgoing[source].append(target)
        incoming[target].append(source)
    by_t = defaultdict(list)
    for node_id, values in nodes.items():
        by_t[values[0]].append(node_id)
    added = []
    for t in sorted(by_t):
        left_ids = [node_id for node_id in by_t[t] if not outgoing[node_id]]
        right_ids = [node_id for node_id in by_t.get(t + 1, []) if not incoming[node_id]]
        for source, target in hungarian_pairs(left_ids, right_ids, nodes, gap_max_um):
            added.append((source, target))
            outgoing[source].append(target)
            incoming[target].append(source)
    return links + added


def safe_divisions(nodes, links):
    outgoing = defaultdict(list)
    incoming = defaultdict(list)
    for source, target in links:
        outgoing[source].append(target)
        incoming[target].append(source)
    by_t = defaultdict(list)
    for node_id, values in nodes.items():
        by_t[values[0]].append(node_id)
    added = []
    for t in sorted(by_t):
        unmatched = [node_id for node_id in by_t.get(t + 1, []) if not incoming[node_id]]
        if not unmatched:
            continue
        for parent in by_t[t]:
            children = outgoing[parent]
            if len(children) >= 2:
                continue
            parent_xyz = nodes[parent]
            candidates = []
            for child in unmatched:
                if dist_um(parent_xyz, nodes[child]) <= div_parent_max_um:
                    candidates.append(child)
            if len(children) == 1:
                sister = children[0]
                candidates = [
                    child
                    for child in candidates
                    if dist_um(nodes[sister], nodes[child]) <= sister_max_um
                ]
                if not candidates:
                    continue
                child = min(candidates, key=lambda c: dist_um(parent_xyz, nodes[c]))
                added.append((parent, child))
                outgoing[parent].append(child)
                incoming[child].append(parent)
                unmatched.remove(child)
            elif len(children) == 0 and len(candidates) >= 2:
                ranked = sorted(candidates, key=lambda c: dist_um(parent_xyz, nodes[c]))
                first, second = ranked[0], None
                for child in ranked[1:]:
                    if dist_um(nodes[first], nodes[child]) <= sister_max_um:
                        second = child
                        break
                if second is None:
                    continue
                added.extend([(parent, first), (parent, second)])
                outgoing[parent].extend([first, second])
                incoming[first].append(parent)
                incoming[second].append(parent)
                unmatched.remove(first)
                unmatched.remove(second)
    return links + added


def prune_short_tracks(nodes, links):
    undirected = defaultdict(set)
    outgoing = defaultdict(list)
    for source, target in links:
        undirected[source].add(target)
        undirected[target].add(source)
        outgoing[source].append(target)
    for node_id in nodes:
        undirected[node_id]
    seen = set()
    keep_nodes = set()
    for node_id in nodes:
        if node_id in seen:
            continue
        stack = [node_id]
        component = []
        seen.add(node_id)
        while stack:
            current = stack.pop()
            component.append(current)
            for neighbor in undirected[current]:
                if neighbor not in seen and neighbor in nodes:
                    seen.add(neighbor)
                    stack.append(neighbor)
        has_division = any(len(outgoing[n]) >= 2 for n in component)
        if has_division or len(component) >= min_track_len:
            keep_nodes.update(component)
    nodes = {node_id: values for node_id, values in nodes.items() if node_id in keep_nodes}
    links = [
        (source, target)
        for source, target in links
        if source in keep_nodes and target in keep_nodes
    ]
    return nodes, links


summary = []
for geff in sorted(pred_dir.glob("*.geff")):
    nodes, links = load_geff(geff)
    before = (len(nodes), len(links))
    links = gap_close(nodes, links)
    links = safe_divisions(nodes, links)
    nodes, links = prune_short_tracks(nodes, links)
    out_path = out_dir / f"{geff.stem}.json"
    payload = {
        "nodes": [
            {"node_id": node_id, "t": t, "z": z, "y": y, "x": x}
            for node_id, (t, z, y, x) in sorted(nodes.items())
        ],
        "edges": [
            {"source_id": source, "target_id": target}
            for source, target in links
        ],
    }
    out_path.write_text(json.dumps(payload))
    summary.append((geff.stem, before[0], before[1], len(nodes), len(links)))

print(json.dumps(summary))
"""

result = subprocess.run(
    [
        sys.executable,
        "-c",
        repair_code,
        str(PRED_DIR),
        str(REPAIR_DIR),
        str(GAP_MAX_UM),
        str(DIVISION_PARENT_MAX_UM),
        str(SISTER_MAX_UM),
        str(MIN_TRACK_LEN),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout.strip())
REPAIR_DIR, len(list(REPAIR_DIR.glob("*.json")))

## Submission

In [ ]:
import csv
import json
from pathlib import Path

rows = []
for path in sorted(REPAIR_DIR.glob("*.json")):
    payload = json.loads(path.read_text())
    name = path.stem
    for node in payload["nodes"]:
        rows.append(
            [
                name,
                "node",
                int(node["node_id"]),
                int(node["t"]),
                int(round(float(node["z"]))),
                int(round(float(node["y"]))),
                int(round(float(node["x"]))),
                -1,
                -1,
            ]
        )
    for edge in payload["edges"]:
        rows.append(
            [
                name,
                "edge",
                -1,
                -1,
                -1,
                -1,
                -1,
                int(edge["source_id"]),
                int(edge["target_id"]),
            ]
        )

with Path(OUTPUT_PATH).open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(
        [
            "id",
            "dataset",
            "row_type",
            "node_id",
            "t",
            "z",
            "y",
            "x",
            "source_id",
            "target_id",
        ]
    )
    for index, row in enumerate(rows):
        writer.writerow([index, *row])

len(rows), OUTPUT_PATH


## Submission Checks

In [ ]:
import csv
from pathlib import Path

expected = set(test_stems)
datasets = set()
row_count = 0
with Path(OUTPUT_PATH).open(encoding="utf-8") as file:
    reader = csv.DictReader(file)
    assert reader.fieldnames[0] == "id"
    for expected_id, row in enumerate(reader):
        assert int(row["id"]) == expected_id
        datasets.add(row["dataset"])
        row_count += 1

assert datasets == expected
assert row_count > 0
row_count, Path(OUTPUT_PATH).stat().st_size